4a) This algorithm creates an augmented matrix with the input matrix A on the left side, and the p x p identity matrix, attempting to reduce the left hand side to the identity matrix to yield A inverse on the right. For every row j in A, it looks for the biggest element in the j'th column in a row greater than or equal to the current row j. It selects this element as the pivot to swap with the current row, and then divides the current row by the inverse so its' normalized to be 1 (we have a 1 in the jth, jth position in the left side matrix). Then, we zero out all column j values in all the other rows in matrix. 


In [17]:
# 4b)
import numpy as np

def gauss_jordan_inverse(A: np.array):
    ROWS, COLS = A.shape
    if ROWS != COLS:
        raise ValueError("Invalid input: A is not square")
    
    identity_matrix = np.eye(ROWS)
    augmented_mat = np.hstack((A, identity_matrix))

    for col_idx in range(COLS):

        # Pivot Selection
        pivot_row = max(range(col_idx, ROWS), key=lambda r: abs(augmented_mat[r, col_idx]))
        if abs(augmented_mat[pivot_row, col_idx]) < 1e-12:
            raise ValueError("Matrix is singular")
        if pivot_row != col_idx:
            augmented_mat[[col_idx, pivot_row]] = augmented_mat[[pivot_row, col_idx]]

        # Normalization
        pivot_val = augmented_mat[col_idx, col_idx]
        augmented_mat[col_idx, :] /= pivot_val
        
        # Eliminate column j value from all other rows
        for row_idx in range(ROWS):
            if row_idx != col_idx:
                factor = augmented_mat[row_idx, col_idx]
                augmented_mat[row_idx, :] -= factor * augmented_mat[col_idx, :]
    
    return augmented_mat[:, COLS:]


In [12]:
# 4c)
B = np.array([[1, 3, 3],
              [1, 4, 3],
              [1, 3, 4]])


B_inv = gauss_jordan_inverse(B)

print("Inverse of B:")
print(B_inv)

identity_check = np.dot(B, B_inv)
print("\nB * B_inv (should be identity):")
print(identity_check)

Inverse of B:
[[ 7. -3. -3.]
 [-1.  1.  0.]
 [-1.  0.  1.]]

B * B_inv (should be identity):
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


In [18]:
#4d)

C = np.array([[1, 2], 
            [2, 4]])

# should be an error, singular matrix
C_inv = gauss_jordan_inverse(C)

ValueError: Matrix is singular

Question 5

In [20]:
# Write a Python function ‘create_ill_conditioned_matrix(p, cond_number)‘ that generates a random
# p → p symmetric, ill-conditioned matrix with a specified condition number. (Hint: Generate a ran-
# dom orthogonal matrix Q. Create a diagonal matrix D with one eigenvalue being 1, another being
# ‘1/cond_number‘, and the rest randomly spaced in between. Then form the matrix A = QDQ ↓ ).

def create_ill_conditioned_matrix(p, cond_number):
    def random_orthogonal(n):
        A = np.random.randn(n, n)
        Q, R = np.linalg.qr(A)
        
        D = np.diag(np.sign(np.diag(R)))
        Q = Q @ D
        
        return Q
    
    orthogonal_matrix = random_orthogonal(p)
    diagonal_matrix = np.eye(p)
    
    diagonal_matrix[0, 0] = 1
    diagonal_matrix[1, 1] = 1 / cond_number


    for diagonal_idx in range(2, p):
        diagonal_matrix[diagonal_idx, diagonal_idx] = np.random.uniform(1/cond_number, 1)
    
    return orthogonal_matrix @ diagonal_matrix @ orthogonal_matrix.T

In [22]:
# (b) Generate a 10 → 10 matrix A with a condition number of 10^8 . Compute its inverse, A ↑ 1 

A = create_ill_conditioned_matrix(10, 10 ** 8)
A_inverse = gauss_jordan_inverse(A)
print(A_inverse)


[[  8930475.92371438   5185080.14794114  -8599164.29131032
  -11959922.96274622   6714657.55864579  -9713165.78485396
  -15408220.38258421  -4491639.08155234   9828946.9424371
   -8706039.71782959]
 [  5185080.14794115   3010488.59110754  -4992721.53660734
   -6943993.97244579   3898564.654467    -5639515.71545588
   -8946093.48751906  -2607871.46489518   5706737.17120113
   -5054771.26309389]
 [ -8599164.29131032  -4992721.53660734   8280148.51252382
   11516223.87523807  -6465551.25864766   9352819.90507903
   14836593.96236992   4325006.40994566  -9464304.56940548
    8383055.79981767]
 [-11959922.96274622  -6943993.97244579  11516223.87523806
   16017040.49946416  -8992442.57911431  13008122.89504496
   20635086.93710822   6015318.96041801 -13163179.70842259
   11659353.77723836]
 [  6714657.55864579   3898564.65446699  -6465551.25864766
   -8992442.57911431   5048627.59554832  -7303148.7897178
  -11585154.22867235  -3377179.77455886   7390202.53487968
   -6545909.19222968]
 [ -971

In [24]:
# c) Create a tiny random perturbation matrix E of the same size, with entries drawn from a uniform
# distribution between ↑ 10 ↑ 9 and 10 ↑ 9 . Create the perturbed matrix ˜A = A + E.

E = np.random.uniform(-10**(-9), 10** (-9), size = (10, 10))

perturbed_A = A + E

In [25]:
# (d) Compute the inverse of the perturbed matrix, ˜A ↑ 1

peturbed_A_inverse = gauss_jordan_inverse(perturbed_A)

In [28]:
# (e) Calculate and print the following:
# • The Frobenius norm of the perturbation: ↔ E ↔ F .
# • The relative error in the input: ↔ E ↔ F / ↔ A ↔ F .
# • The Frobenius norm of the change in the inverse: ↔ A ↑ 1 ↑ ˜A ↑ 1 ↔ F .
# • The relative error in the output: ↔ A ↑ 1 ↑ ˜A ↑ 1 ↔ F / ↔ A ↑ 1 ↔ F .

#The Frobenius norm of the perturbation
perturbation_frobenius_norm = np.linalg.norm(E)

A_frobenius_norm = np.linalg.norm(A)

# The relative error in the input
relative_input_error = perturbation_frobenius_norm / A_frobenius_norm


inverse_differences = A_inverse - peturbed_A_inverse

# The Frobenius norm of the change in the inverse
inverse_differences_frobenius_norm = np.linalg.norm(inverse_differences)


# The relative error in the output

A_inverse_frobenius_norm = np.linalg.norm(A_inverse)
relative_output_error = inverse_differences_frobenius_norm / A_inverse_frobenius_norm



print(f"Frobenius norm of perturbation (||E||_F): {perturbation_frobenius_norm}")
print(f"Relative error in input (||E||_F / ||A||_F): {relative_input_error}")
print(f"Frobenius norm of change in inverse (||A^-1 - Ã^-1||_F): {inverse_differences_frobenius_norm}")
print(f"Relative error in output (||A^-1 - Ã^-1||_F / ||A^-1||_F): {relative_output_error}")

Frobenius norm of perturbation (||E||_F): 5.853903272891288e-09
Relative error in input (||E||_F / ||A||_F): 2.710383563505233e-09
Frobenius norm of change in inverse (||A^-1 - Ã^-1||_F): 10612443.175564304
Relative error in output (||A^-1 - Ã^-1||_F / ||A^-1||_F): 0.1061244310708248


(f)Comment on the results. How does the relative error in the output compare to the relative error in
the input? What does this demonstrate about inverting ill-conditioned matrices?



The relative error in the output is much larger than the relative error in the output. Altering the original
matrix A by the perturbation matrix leads to a < 0.000001 % difference in the input matrix A. This extremely small
perturbation leads to an over 10% relative change in the inverted matrix A_inv between A and perturbed_A. What this means is that 
you must analyze a matrix condition number before using its numerical inverse -- if the condition number is large, small measurement errors or
numerical imprecisions caused by insufficient bit representations can cause extreme differences in the matrix inverse, which can cause issues
for reliable inference to the problem inverses are being used for. 